## Great, now that we discussed a little let's continue

Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


- using nf-core pipelines which provide standardized and version-controlled workflows
- running the tools in docker container ensures consistent and reproducible environment

You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

- Pipeline: rnaseq

Analyze the 2 files using an nf-core pipeline.

What does this pipeline do?

Which are the main tools that will be used in the pipeline?

- analyses RNA sequencing data with a reference genome and respective annotation

- quality control, trimming and (pseudo-)alignment

- produces gene expression matrix and QC report

- main tools:
    - FastQC
    - Trim Galore
    - STAR/HiSAT2
    - SAMtools
    - Picard
    - DESeq2

As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

In [2]:
# Creating the sample sheet
## sample,fastq_1,fastq_2,strandedness,seq_platform
## CONTROL_REP1,AEG588A1_S1_L002_R1_001.fastq.gz,AEG588A1_S1_L002_R2_001.fastq.gz,auto,ILLUMINA
import pandas as pd
sample_sheet_raw = pd.read_csv("results_fetch/samplesheet/samplesheet.csv")
#print(sample_sheet_raw.head())

sample_sheet = sample_sheet_raw[["sample", "fastq_1", "fastq_2"]].copy()
sample_sheet["fastq_1"] = sample_sheet["fastq_1"].str.replace(
    "./SRA_runs/fastq/",
    "./results_fetch/fastq/",
    regex=False
)

sample_sheet["fastq_2"] = sample_sheet["fastq_2"].str.replace(
    "./SRA_runs/fastq/",
    "./results_fetch/fastq/",
    regex=False
)
sample_sheet["strandedness"] = "auto"
#sample_sheet["seq_platform"] = "ILLUMINA"
print(sample_sheet.head())

sample_sheet.to_csv("results_fetch/samplesheet/samplesheet.csv", index=False)

        sample                                            fastq_1  \
0  SRX19144486  ./results_fetch/fastq/SRX19144486_SRR23195516_...   
1  SRX19144488  ./results_fetch/fastq/SRX19144488_SRR23195511_...   

                                             fastq_2 strandedness  
0  ./results_fetch/fastq/SRX19144486_SRR23195516_...         auto  
1  ./results_fetch/fastq/SRX19144488_SRR23195511_...         auto  


In [ ]:
# post here the command you used to run nf-core/rnaseq
import os.path

if not os.path.isfile("results_rnaseq/samplesheets/samplesheet_with_bams.csv"):
    print("Create bams")
    !nextflow run nf-core/rnaseq -profile docker -params-file params_rnaseq_01.yaml -c nextflow.config -resume
else:
    !nextflow run nf-core/rnaseq -profile docker -params-file params_rnaseq_02.yaml -c nextflow.config -resume

Create bams
Nextflow 26.10.0 is available - Please consider updating your version to it

 N E X T F L O W   ~  version 26.04.7

NOTE: Your local project version looks outdated - a different revision is available in the remote repository [95dbcd977a]
Launching `https://github.com/nf-core/rnaseq` [silly_einstein] revision: a1fcdddd3b [master]

WARN: Unrecognized config option 'manifest.diagram'

------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/rnaseq 3.27.0
------------------------------------------------------

Input/output options
  input               : results_fetch/samplesheet/samplesheet.csv
  outdir              : results_rnaseq

Reference genome options
  fasta               : https://ftp.ensembl.org/pub/release-102/fasta/m

IOStream.flush timed out



executor >  local (14)
[]8;;file:///home/laura/CW_BMD/computational-workflows-2026/day_02/work/7a/d5150921b979b695ae2f6e2f48ec0d7a/d51509]8;;] NFC…usculus.GRCm38.102.gtf.gz) | 1 of 1, cached: 1 ✔
[]8;;file:///home/laura/CW_BMD/computational-workflows-2026/day_02/work/d4/96e68ee14a9e549d4dfef46ed803c5d4/96e68e]8;;] NFC…8.dna.chromosome.19.fa.gz) | 1 of 1, cached: 1 ✔
[]8;;file:///home/laura/CW_BMD/computational-workflows-2026/day_02/work/32/adf8b0fb7a27e67785c2342d8bfc6f32/adf8b0]8;;] NFC…culus.GRCm38.102.filtered) | 1 of 1 ✔
[]8;;file:///home/laura/CW_BMD/computational-workflows-2026/day_02/work/9a/f7c1d60ecc970bc83e7a24303f8c7b9a/f7c1d6]8;;] NFC…culus.GRCm38.102.filtered) | 1 of 1 ✔
[]8;;file:///home/laura/CW_BMD/computational-workflows-2026/day_02/work/ac/75264bb864409b9258a0e2b0ee301bac/75264b]8;;] NFC…Cm38.dna.chromosome.19.fa) | 1 of 1 ✔
[]8;;file:///home/laura/CW_BMD/computational-workflows-2026/day_02/work/85/d68b0d83fcf78811a2dfdbdbdb02e885/d68b0d]8;;] 

Explain all the parameters you set and why you set them in this way.



- profile: to use docker containers
- input: input samplesheet
- save_align_intermeds: to save created bam files from first successful run for efficent reprocessing
- outdir: results Directory
- gtf: link to the transcriptome reference
- fasta: link to the reference genome
- aligner: defining a specific alignment tool
- min_mapped_reads: Minimum number of reads that must map successfully to the reference genome for a sample to pass the mapping quality control threshold -> 0 to not crash the pipeline
- skip_alignment: to skip the alignment in case of reprocessing

## Browsing the results

How did the pipeline perform?

- performed well but took an extensive amount of time to align the reads to reference genome

Explain the quality control steps. Are you happy with the quality and why. If not, why not.
- FastQC: assessing per-base sequencing quality for raw and trimmed reads
- Trim Galore: removes adapter sequences and low-quality bases
- DupRadar: rating duplication rate to read count
- Mapping statistics during alignment: assessing quality of read alignment
- Qualimap: evaluates alignment quality and sequencing coverage
- Samtools: generates alignment statistics from BAM files, including the number of mapped and unmapped reads, mapping rates, and other alignment metrics
- RSeQC: assesses RNA-seq-specific quality metrics, such as read distribution across genomic features, gene body coverage, transcript integrity, and strand specificity
- MultiQC: combines quality metrices from different tools


- overall good quality, like read length, GC content, Phred score, read counts per sample etc

Please give additional information on : 
- ribosomal rRNA: unwanted component when analysing mRNA expression changes
- Duplication: amount of duplicated reads with the same sequence in the sample
- GC content: amount of G and C bases in the sequences, affecting stability and unusual distributions can indicate contamination or sequencing bias

What are the possible steps that could lead to poorer results?
- incorrect read trimming removing RNA-sequence
- wrong reference Genome/transcriptomic reference
- low sequencing Quality and low sequencing depth

Would you exclude any samples? If yes, which and why?

- no

What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?

HOW TO CONTINUE:
- statistical comparison of differential gene expression between the groups
- biological interpretation: identify affected pathways and functional enrichments

SCIENTIFIC QUESTION:
- how oxycodone withdrawal affects gene expression in the brain especially in the context of chronic pain
- if HDAC1/2 contribute to the expression changes

PACKAGES:
- DESeq2, PyDESeq2
- ClusterProfiler